In [11]:
from rdflib import Graph, RDF, RDFS, OWL, URIRef, BNode, Literal, Namespace
import pandas as pd
from tqdm import tqdm

# -------------------- NAMESPACES & CONFIG --------------------
OIO = Namespace("http://www.geneontology.org/formats/oboInOwl#")
IAO = Namespace("http://purl.obolibrary.org/obo/IAO_")
CL_PREFIX = "http://purl.obolibrary.org/obo/CL_"
UBERON_PREFIX = "http://purl.obolibrary.org/obo/UBERON_"
HPO_PREFIX = "http://purl.obolibrary.org/obo/HP_"

class HPOExtractor:
    def __init__(self, hpo_g, cl_g=None, uberon_g=None):
        self.hpo_g = hpo_g
        self.cl_g = cl_g
        self.uberon_g = uberon_g
        self.cache = {}
        self.obsolete_maps = {
            "CL": self._build_enhanced_obsolete_map(cl_g) if cl_g else {},
            "UBERON": self._build_enhanced_obsolete_map(uberon_g) if uberon_g else {}
        }

    def _build_enhanced_obsolete_map(self, g):
        obs = {}
        for cls in g.subjects(RDF.type, OWL.Class):
            if (cls, OWL.deprecated, Literal(True)) in g:
                replacement = g.value(cls, OIO.replacedBy) or g.value(cls, IAO["0100001"])
                if replacement:
                    obs[str(cls)] = str(replacement)
        return obs

    def get_label(self, g, term):
        if not g or not term: return None
        label = g.value(term, RDFS.label)
        if label: return str(label)
        if isinstance(term, URIRef): return str(term).split("/")[-1]
        return "Anonymous_Node"

    def extract(self, node, hpo_iri, hpo_label, follow_hierarchy=True, path=None):
        """
        path: Tracks the current recursion branch to prevent infinite loops 
              WITHOUT blocking other phenotypes from visiting the same parents.
        """
        if path is None: path = set()
        
        # 1. Branch-specific loop prevention
        if node in path: return {"CL": [], "UBERON": []}
        
        # 2. Global Cache (Only for Named Classes, never for BNodes)
        node_str = str(node)
        if isinstance(node, URIRef) and node_str in self.cache and follow_hierarchy:
            # We must re-map the cached results to the CURRENT HPO term
            return self._remap_cached_results(self.cache[node_str], hpo_iri, hpo_label)

        # Add to current recursion path
        new_path = path | {node}
        local_records = {"CL": [], "UBERON": []}

        # 3. Extract Restrictions (The Core Logic)
        # Using a more manual triple check to ensure we aren't missing anything
        for p, o in self.hpo_g.predicate_objects(node):
            # Handle Some/All Values From
            if p in [OWL.someValuesFrom, OWL.allValuesFrom]:
                logic_name = "SOME" if p == OWL.someValuesFrom else "ALL"
                prop = self.hpo_g.value(node, OWL.onProperty)
                filler_iri = str(o)
                
                # Check for CL/UBERON Prefixes
                target = None
                if filler_iri.startswith(CL_PREFIX): target = "CL"
                elif filler_iri.startswith(UBERON_PREFIX): target = "UBERON"

                if target:
                    graph = self.cl_g if target == "CL" else self.uberon_g
                    local_records[target].append(self._create_record(
                        hpo_iri, hpo_label, logic_name, prop, filler_iri, graph, target
                    ))
                
                # If the filler is a BNode, dive in
                if isinstance(o, BNode):
                    nested = self.extract(o, hpo_iri, hpo_label, follow_hierarchy, new_path)
                    for k in local_records: local_records[k].extend(nested[k])

            # 4. Handle Logical Operators (Intersections/Unions)
            if p in [OWL.intersectionOf, OWL.unionOf]:
                logic_name = "AND" if p == OWL.intersectionOf else "OR"
                # Use rdflib's Collection helper to get the items in the list
                for item in self.hpo_g.items(o):
                    nested = self.extract(item, hpo_iri, hpo_label, follow_hierarchy, new_path)
                    for k in nested:
                        for row in nested[k]:
                            row["Logic"] = f"{logic_name}->{row['Logic']}"
                        local_records[k].extend(nested[k])

        # 5. Inheritance (Climb the tree)
        if follow_hierarchy:
            for parent in self.hpo_g.objects(node, RDFS.subClassOf):
                # Only climb if it's a real class or a restriction
                if isinstance(parent, (URIRef, BNode)):
                    inherited = self.extract(parent, hpo_iri, hpo_label, True, new_path)
                    for k in local_records: local_records[k].extend(inherited[k])
            
            for parent in self.hpo_g.objects(node, OWL.equivalentClass):
                if isinstance(parent, (URIRef, BNode)) and parent != node:
                    inherited = self.extract(parent, hpo_iri, hpo_label, True, new_path)
                    for k in local_records: local_records[k].extend(inherited[k])

        # 6. Save to Cache (Named classes only)
        if isinstance(node, URIRef) and follow_hierarchy:
            self.cache[node_str] = local_records

        return local_records

    def _remap_cached_results(self, cached_list, new_iri, new_label):
        """Creates a copy of cached results updated with the current HPO term's info."""
        new_results = {"CL": [], "UBERON": []}
        for k in ["CL", "UBERON"]:
            for item in cached_list[k]:
                new_item = item.copy()
                new_item["HPO_IRI"] = str(new_iri)
                new_item["HPO_Label"] = new_label
                new_results[k].append(new_item)
        return new_results

    def _create_record(self, hpo_iri, hpo_label, logic, prop, filler_iri, graph, target):
        is_obs = filler_iri in self.obsolete_maps[target]
        repl_iri = self.obsolete_maps[target].get(filler_iri)
        return {
            "HPO_IRI": str(hpo_iri),
            "HPO_Label": hpo_label,
            "Logic": logic,
            "Property": self.get_label(self.hpo_g, prop),
            "Filler_IRI": filler_iri,
            "Filler_Label": self.get_label(graph, URIRef(filler_iri)),
            "Obsolete": is_obs,
            "Replacement_IRI": repl_iri,
            "Replacement_Label": self.get_label(graph, URIRef(repl_iri)) if repl_iri else None
        }

# -------------------- EXECUTION BLOCK --------------------
def main():
    # 1. Load Ontologies
    print("Loading ontologies (this may take a minute)...")
    hpo_g = Graph().parse("../data/hp.owl", format="xml")
    cl_g = Graph().parse("../data/cl.owl", format="xml")
    uberon_g = Graph().parse("../data/uberon.owl", format="xml")

    # 2. Initialize Extractor
    extractor = HPOExtractor(hpo_g, cl_g, uberon_g)
    
    # 3. Collect HPO terms
    hpo_classes = [c for c in hpo_g.subjects(RDF.type, OWL.Class) 
                   if str(c).startswith(HPO_PREFIX)]

    all_cl, all_uberon = [], []

    # 4. Run loop with progress bar
    for cls in tqdm(hpo_classes, desc="Extracting Mappings"):
        label = extractor.get_label(hpo_g, cls)
        results = extractor.extract(cls, cls, label, follow_hierarchy=True)
        all_cl.extend(results["CL"])
        all_uberon.extend(results["UBERON"])

    # 5. Save to CSV
    df_cl = pd.DataFrame(all_cl).drop_duplicates()
    df_uberon = pd.DataFrame(all_uberon).drop_duplicates()

    df_cl.to_csv("hpo_to_cl_mappings.csv", index=False)
    df_uberon.to_csv("hpo_to_uberon_mappings.csv", index=False)
    
    print(f"\nSuccess! Found {len(df_cl)} cell links and {len(df_uberon)} anatomy links.")
    return df_cl, df_uberon

if __name__ == "__main__":
    df_cl, df_uberon = main()

Loading ontologies (this may take a minute)...


Extracting Mappings: 100%|██████████| 19726/19726 [00:01<00:00, 15042.11it/s]



Success! Found 14377 cell links and 178419 anatomy links.


In [12]:
df_cl_no_cell = df_cl[~df_cl['Filler_IRI'].str.contains("http://purl.obolibrary.org/obo/CL_0000000", case=False, na=False)]

In [13]:
df_cl_no_cell

,HPO_IRI,HPO_Label,Logic,Property,Filler_IRI,Filler_Label,Obsolete,Replacement_IRI,Replacement_Label
0,http://purl.obolibrary.org/obo/HP_0000027,Azoospermia,AND->AND->SOME,develops from,http://purl.obolibrary.org/obo/CL_0000018,spermatid,False,None,None
2,http://purl.obolibrary.org/obo/HP_0000235,Abnormal cranial suture/fontanelle morphology,AND->OR->SOME,has part,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,False,None,None
4,http://purl.obolibrary.org/obo/HP_0000235,Abnormal cranial suture/fontanelle morphology,AND->OR->AND->SOME,has part,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,False,None,None
8,http://purl.obolibrary.org/obo/HP_0000235,Abnormal cranial suture/fontanelle morphology,AND->OR->OR->SOME,has part,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,False,None,None
10,http://purl.obolibrary.org/obo/HP_0000235,Abnormal cranial suture/fontanelle morphology,AND->OR->OR->AND->SOME,has part,http://purl.obolibrary.org/obo/CL_0002320,connective tissue cell,False,None,None
...,...,...,...,...,...,...,...,...,...
109933,http://purl.obolibrary.org/obo/HP_6001239,Reduced circulating reticulated platelet count,AND->SOME,characteristic of part of,http://purl.obolibrary.org/obo/CL_0000233,platelet,False,None,None
109934,http://purl.obolibrary.org/obo/HP_6001239,Reduced circulating reticulated platelet count,AND->SOME,characteristic of,http://purl.obolibrary.org/obo/CL_0000233,platelet,False,None,None
109936,http://purl.obolibrary.org/obo/HP_6001240,Abnormal circulating reticulated platelet count,AND->SOME,characteristic of part of,http://purl.obolibrary.org/obo/CL_0000233,platelet,False,None,None
109937,http://purl.obolibrary.org/obo/HP_6001240,Abnormal circulating reticulated platelet count,AND->SOME,characteristic of,http://purl.obolibrary.org/obo/CL_0000233,platelet,False,None,None
